# Lipizone dissection table generator

Generates virtual section dissection tables from the Allen CCFv3 annotation volume,
colored by lipizone classes (lev3). Includes a three-stage spatial cleanup pipeline
(surrounded-by relabeling → small-region merging → iterative connected-component merging)
to produce contiguous dissection territories.

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"]       = "1"
os.environ["MKL_NUM_THREADS"]       = "1"
os.environ["NUMEXPR_NUM_THREADS"]   = "1"

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from tqdm import tqdm
from bg_atlasapi import BrainGlobeAtlas

matplotlib.rcParams['pdf.fonttype'] = 42

## Load data

In [ ]:
atlas = pd.read_parquet("./zenodo/maindata_2.parquet") # download from: https://zenodo.org/records/15379565
atlas = atlas.loc[atlas['Condition'] == "naive", :].copy()

aaa = BrainGlobeAtlas("allen_mouse_25um")
allen_ids = aaa.annotation

## Global prep: lev3 labels, id→lev3, lev3→color

In [ ]:
# Composite lev3 label per row
atlas['lev3'] = (
    atlas['level_1'].astype(str)
    + atlas['level_2'].astype(str)
    + atlas['level_3'].astype(str)
)

# Keep only acronyms with >= 100 voxels
acronym_counts = atlas['acronym'].value_counts()
valid_acronyms = acronym_counts[acronym_counts >= 100].index
atlas = atlas[atlas['acronym'].isin(valid_acronyms)].copy()

# Modal lev3 per acronym
best_l3_per_acronym = (
    atlas
    .groupby('acronym')['lev3']
    .agg(lambda s: s.value_counts().idxmax())
)

# acronym → id (1:1), then id → lev3
acr_to_id = (
    atlas
    .drop_duplicates('acronym')
    .set_index('acronym')['id']
)

id_to_l3 = {
    acr_to_id[acr]: l3
    for acr, l3 in best_l3_per_acronym.items()
}

# Modal lipizone color per lev3
l3_to_color = (
    atlas
    .dropna(subset=['lev3', 'lipizone_color'])
    .groupby('lev3')['lipizone_color']
    .agg(lambda s: s.value_counts().idxmax())
    .to_dict()
)

## Virtual sectioning

In [ ]:
def make_section_from_volume(allen_ids, axis='z', index=0):
    """
    Extract a 2D section from the 3D annotation volume (Z, Y, X).

    Parameters
    ----------
    allen_ids : array-like, shape (Z, Y, X)
    axis : str, one of 'z' (coronal), 'y' (sagittal), 'x' (horizontal)
    index : int, slice position along the chosen axis

    Returns
    -------
    DataFrame with columns x, y, id. Background voxels (id == 0) set to NaN.
    """
    allen_ids = np.asarray(allen_ids)
    Z, Y, X = allen_ids.shape

    if axis == 'z':
        assert 0 <= index < Z, f"index out of range for Z: 0 <= index < {Z}"
        slice_ids = allen_ids[index, :, :]
        yy, xx = np.mgrid[0:Y, 0:X]
    elif axis == 'y':
        assert 0 <= index < Y, f"index out of range for Y: 0 <= index < {Y}"
        slice_ids = allen_ids[:, index, :]
        yy, xx = np.mgrid[0:Z, 0:X]
    elif axis == 'x':
        assert 0 <= index < X, f"index out of range for X: 0 <= index < {X}"
        slice_ids = allen_ids[:, :, index]
        zz, yy = np.mgrid[0:Z, 0:Y]
        xx = yy
        yy = zz
    else:
        raise ValueError("axis must be one of 'x', 'y', 'z'")

    df = pd.DataFrame({'x': xx.ravel(), 'y': yy.ravel(), 'id': slice_ids.ravel()})
    df.loc[df['id'] == 0, 'id'] = np.nan
    return df

## Section cleanup pipeline

Three stages applied per section:
1. **Surrounded-by relabeling** — if all 4-connected neighbors of a region share the same lev3, relabel it.
2. **Small-region merging** — regions smaller than `min_pixels` are merged into their largest border neighbor,
   unless a same-lev3 large neighbor already exists.
3. **Iterative connected-component merging** — spatially disconnected fragments of the same lev3 that are
   smaller than `min_comp_pixels` are absorbed into the dominant neighboring lev3.

In [ ]:
def _four_connected_neighbor_ids(pix, id_col='id'):
    """Return DataFrame of (id, neighbor_id) pairs from a pixel grid indexed by (x, y)."""
    adj_list = []
    for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
        neighbor = pix[id_col].copy()
        neighbor.index = pd.MultiIndex.from_tuples(
            [(ix[0] + dx, ix[1] + dy) for ix in neighbor.index],
            names=['x', 'y']
        )
        df_tmp = pd.DataFrame({
            id_col: pix[id_col],
            f'neighbor_{id_col}': neighbor.reindex(pix.index)
        })
        df_tmp = df_tmp[
            df_tmp[f'neighbor_{id_col}'].notna() &
            (df_tmp[f'neighbor_{id_col}'] != df_tmp[id_col])
        ]
        adj_list.append(df_tmp)
    if adj_list:
        return pd.concat(adj_list, ignore_index=True).drop_duplicates()
    return pd.DataFrame(columns=[id_col, f'neighbor_{id_col}'])


def process_section(section, id_to_l3, l3_to_color,
                    min_pixels=1000, min_comp_pixels=1000):
    """
    Run the full cleanup pipeline on a section DataFrame (columns: x, y, id).
    Returns a copy with 'l3_final' and 'color_final' columns.
    """
    sec = section.copy()

    # --- Stage 1: initial lev3 assignment ---
    sec['l3'] = sec['id'].map(id_to_l3)

    # --- Stage 2: surrounded-by relabeling (ID-level) ---
    pix = (
        sec[sec['id'].notna()]
        .set_index(['x', 'y'])[['id', 'l3']]
        .copy()
    )
    id_to_l3_section = pix.reset_index().groupby('id')['l3'].first()

    adj_pairs = _four_connected_neighbor_ids(pix, id_col='id')

    if not adj_pairs.empty:
        adj_pairs['neighbor_l3'] = adj_pairs['neighbor_id'].map(id_to_l3_section)
        homog_neighbors = (
            adj_pairs
            .groupby('id')['neighbor_l3']
            .agg(lambda s: s.iloc[0] if s.nunique() == 1 else np.nan)
            .dropna()
        )
    else:
        homog_neighbors = pd.Series(dtype=object)

    id_to_l3_cleaned = id_to_l3_section.copy()
    id_to_l3_cleaned.update(homog_neighbors)
    sec['l3_cleaned'] = sec['id'].map(id_to_l3_cleaned)

    # --- Stage 3: merge small IDs by border contact ---
    pixel_counts = sec.loc[sec['id'].notna(), 'id'].value_counts()
    small_ids = pixel_counts[pixel_counts < min_pixels].index

    id_to_l3_cleaned = (
        sec[sec['id'].notna()]
        .groupby('id')['l3_cleaned']
        .first()
    )

    pix = (
        sec[sec['id'].notna()]
        .set_index(['x', 'y'])[['id']]
        .copy()
    )
    border_contacts = _four_connected_neighbor_ids(pix, id_col='id')

    if not border_contacts.empty:
        border_counts = (
            border_contacts
            .groupby(['id', 'neighbor_id'])
            .size()
            .rename('border_pixels')
            .reset_index()
        )
    else:
        border_counts = pd.DataFrame(columns=['id', 'neighbor_id', 'border_pixels'])

    id_merge_map = {}
    for rid in small_ids:
        sb = border_counts[border_counts['id'] == rid]
        if sb.empty:
            continue
        # Keep if a large same-lev3 neighbor exists
        keep = any(
            pixel_counts.get(nb, 0) >= min_pixels
            and id_to_l3_cleaned.get(nb) == id_to_l3_cleaned.get(rid)
            for nb in sb['neighbor_id'].unique()
        )
        if not keep:
            best_row = sb.loc[sb['border_pixels'].idxmax()]
            id_merge_map[rid] = best_row['neighbor_id']

    sec['id_merged'] = sec['id'].replace(id_merge_map)
    sec['l3_merged'] = sec['id_merged'].map(id_to_l3_cleaned)

    # --- Stage 4: iterative connected-component merging ---
    for iteration in range(50):
        pix = (
            sec[['x', 'y', 'l3_merged']]
            .dropna(subset=['l3_merged'])
            .copy()
        )
        pix['idx'] = np.arange(len(pix))
        pix_xy = pix.set_index(['x', 'y'])
        n = len(pix)
        if n == 0:
            break

        # Build 4-connected edges
        edges = []
        for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            neighbor_idx = pix_xy['idx'].copy()
            neighbor_idx.index = pd.MultiIndex.from_tuples(
                [(ix[0] + dx, ix[1] + dy) for ix in neighbor_idx.index],
                names=['x', 'y']
            )
            df_tmp = pd.DataFrame({
                'idx': pix_xy['idx'],
                'neighbor_idx': neighbor_idx.reindex(pix_xy.index).values
            }).dropna(subset=['neighbor_idx'])
            df_tmp['neighbor_idx'] = df_tmp['neighbor_idx'].astype(int)
            edges.append(df_tmp)

        edges_df = pd.concat(edges, ignore_index=True) if edges else pd.DataFrame(columns=['idx', 'neighbor_idx'])
        if edges_df.empty:
            break

        # Union-Find on same-lev3 edges
        same_l3 = (
            pix['l3_merged'].values[edges_df['idx'].values]
            == pix['l3_merged'].values[edges_df['neighbor_idx'].values]
        )
        edges_same = edges_df[same_l3]

        parent = np.arange(n)

        def find(i):
            while parent[i] != i:
                parent[i] = parent[parent[i]]
                i = parent[i]
            return i

        def union(i, j):
            ri, rj = find(i), find(j)
            if ri != rj:
                parent[rj] = ri

        for i, j in zip(edges_same['idx'].values, edges_same['neighbor_idx'].values):
            union(i, j)

        pix['comp'] = np.array([find(i) for i in range(n)])
        comp_sizes = pix.groupby('comp').size().rename('n_pixels')

        # Border contact frequencies per component
        edges_df['comp'] = pix['comp'].values[edges_df['idx'].values]
        edges_df['neighbor_comp'] = pix['comp'].values[edges_df['neighbor_idx'].values]
        edges_df['neighbor_l3'] = pix['l3_merged'].values[edges_df['neighbor_idx'].values]
        border_edges = edges_df[edges_df['comp'] != edges_df['neighbor_comp']]

        if not border_edges.empty:
            bc = (
                border_edges
                .groupby(['comp', 'neighbor_l3'])
                .size()
                .rename('freq')
                .reset_index()
            )
        else:
            bc = pd.DataFrame(columns=['comp', 'neighbor_l3', 'freq'])

        # Relabel small components to their dominant border neighbor
        small_comps = comp_sizes[comp_sizes < min_comp_pixels].index
        comp_to_new_l3 = {}
        for comp in small_comps:
            rows = bc[bc['comp'] == comp]
            if rows.empty:
                continue
            comp_to_new_l3[comp] = rows.loc[rows['freq'].idxmax(), 'neighbor_l3']

        if not comp_to_new_l3:
            break

        pix['l3_merged_new'] = pix['l3_merged']
        mask_comp = pix['comp'].isin(comp_to_new_l3.keys())
        pix.loc[mask_comp, 'l3_merged_new'] = pix.loc[mask_comp, 'comp'].map(comp_to_new_l3)

        label_df = pix[['x', 'y', 'l3_merged_new']].set_index(['x', 'y'])
        sec = sec.drop(columns=['l3_merged'], errors='ignore')
        sec = sec.join(label_df, on=['x', 'y'])
        sec.rename(columns={'l3_merged_new': 'l3_merged'}, inplace=True)

    # Final columns
    sec['l3_final'] = sec['l3_merged']
    mask = sec['l3_final'].notna()
    sec['color_final'] = np.nan
    sec.loc[mask, 'color_final'] = sec.loc[mask, 'l3_final'].map(l3_to_color)

    return sec

## Generate virtual dissection tables (3 × 10 grid)

In [ ]:
Z, Y, X = allen_ids.shape
n_slices = 10


def central_regular_indices(N, n):
    """Return up to n regularly spaced indices, excluding the first and last."""
    n_full = min(N, n + 2)
    full = np.linspace(0, N - 1, n_full, dtype=int)
    core = full[1:-1] if n_full > 2 else full
    return core[:min(n, len(core))]


z_indices = central_regular_indices(Z, n_slices)
y_indices = central_regular_indices(Y, n_slices)
x_indices = central_regular_indices(X, n_slices)

# Global voxel-based axis limits (consistent across all panels)
global_x_max = max(X, Y)
global_y_max = max(Y, Z)


def set_global_limits(ax):
    ax.set_xlim(0, global_x_max - 1)
    ax.set_ylim(-global_y_max, 0)


def clean_ax(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_aspect('equal', adjustable='box')


fig, axes = plt.subplots(
    3, n_slices,
    figsize=(2 * n_slices, 8),
    gridspec_kw={'height_ratios': [1, 2, 2]}
)
axes = np.atleast_2d(axes)
marker_size = 0.5

for row, (axis_name, indices) in enumerate(
    [('z', z_indices), ('y', y_indices), ('x', x_indices)]
):
    for j in tqdm(range(n_slices), desc=f"{axis_name}-axis"):
        ax = axes[row, j]
        if j < len(indices):
            idx = indices[j]
            section = make_section_from_volume(allen_ids, axis=axis_name, index=idx)
            sec_proc = process_section(section, id_to_l3, l3_to_color,
                                       min_pixels=1000, min_comp_pixels=1000)
            mask = sec_proc['l3_final'].notna()
            ax.scatter(
                sec_proc.loc[mask, 'x'],
                -sec_proc.loc[mask, 'y'],
                c=list(sec_proc.loc[mask, 'color_final']),
                s=marker_size,
                rasterized=True,
            )
            ax.set_title(f"{axis_name} = {idx}", fontsize=8)
        set_global_limits(ax)
        clean_ax(ax)

plt.subplots_adjust(wspace=0, hspace=0)
fig.savefig("dissectiontables.pdf", bbox_inches='tight', pad_inches=0, dpi=300)
plt.show()
plt.close(fig)

## Atlas overview (pre-dissection, all 32 sections)

In [ ]:
data = atlas
fig, axes = plt.subplots(4, 8, figsize=(40, 20))
axes = axes.flatten()
dot_size = 0.3

sections_to_plot = range(1, 33)

global_min_z = data['zccf'].min()
global_max_z = data['zccf'].max()
global_min_y = -data['yccf'].max()
global_max_y = -data['yccf'].min()

for i, section_num in enumerate(sections_to_plot):
    ax = axes[i]
    xx = data[data['SectionID'] == section_num]
    colors = xx['lev3'].map(l3_to_color)

    ax.scatter(
        xx['zccf'], -xx['yccf'],
        c=list(colors),
        s=dot_size,
        rasterized=True
    )
    ax.axis('off')
    ax.set_aspect('equal')
    ax.set_xlim(global_min_z, global_max_z)
    ax.set_ylim(global_min_y, global_max_y)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig("lipizoneclasses_predissection.pdf")
plt.show()

## Before/after comparison (atlas vs. processed)

In [ ]:
data = atlas
all_sections = np.sort(data['SectionID'].unique()[:27])
n_show = min(6, len(all_sections))
section_indices = np.linspace(0, len(all_sections) - 1, n_show, dtype=int)
sections_to_plot = all_sections[section_indices]

global_min_z = data['zccf'].min()
global_max_z = data['zccf'].max()
global_min_y = -data['yccf'].max()
global_max_y = -data['yccf'].min()
dot_size = 0.3

# Process each section
processed_sections = {}
for sid in sections_to_plot:
    sec_orig = data[data['SectionID'] == sid].copy()
    sec_proc = process_section(sec_orig, id_to_l3, l3_to_color,
                               min_pixels=1000, min_comp_pixels=1000)
    processed_sections[sid] = sec_proc

fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))


def clean_ax_ccf(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(global_min_z, global_max_z)
    ax.set_ylim(global_min_y, global_max_y)


# Top row: original atlas
for j, sid in enumerate(sections_to_plot):
    ax = axes[0, j]
    xx = data[data['SectionID'] == sid]
    ax.scatter(
        xx['zccf'], -xx['yccf'],
        c=list(xx['lev3'].map(l3_to_color)),
        s=dot_size, rasterized=True
    )
    ax.set_title(f"Section {sid} (atlas)", fontsize=10)
    clean_ax_ccf(ax)

# Bottom row: processed
for j, sid in enumerate(sections_to_plot):
    ax = axes[1, j]
    sec_proc = processed_sections[sid]
    mask = sec_proc['l3_final'].notna()
    ax.scatter(
        sec_proc.loc[mask, 'zccf'],
        -sec_proc.loc[mask, 'yccf'],
        c=list(sec_proc.loc[mask, 'l3_final'].map(l3_to_color)),
        s=dot_size, rasterized=True
    )
    ax.set_title(f"Section {sid} (processed)", fontsize=10)
    clean_ax_ccf(ax)

plt.tight_layout()
plt.savefig("lipizoneclasses_dissectionready.pdf")
plt.show()